In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr, mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# Plot style
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})

# Load
df = pd.read_csv("manufacturing.csv")
df.drop(columns=["Unnamed: 0"], inplace=True)

# Binary anomaly flag (1 = any breakdown, 0 = none)
df["BREAKDOWN"] = (df["BREAKS"] > 0).astype(int)

print(f"Dataset shape : {df.shape}")
print(f"\nBREAKS distribution:\n{df['BREAKS'].value_counts().sort_index()}")
print(f"\nBreakdown rate: {df['BREAKDOWN'].mean()*100:.1f}%")

Dataset shape : (17600, 24)

BREAKS distribution:
BREAKS
0    12332
1     4199
2     1054
3       15
Name: count, dtype: int64

Breakdown rate: 29.9%


In [ ]:
df.drop_duplicates()
df.duplicated().sum()
df.isna().sum()

,0
ID,0
Priority,0
Family_type,0
First_stage,0
Start_time_S1,0
Finish_time_S1,0
Processing_Time_S1,0
Second_stage,0
Start_time_S2,0
Finish_Time_S2,0


In [ ]:
df.head()

,ID,Priority,Family_type,First_stage,Start_time_S1,Finish_time_S1,Processing_Time_S1,Second_stage,Start_time_S2,Finish_Time_S2,...,Processing_Time_S3,Fourth_stage,Start_time_s4,Finish_time,Processing_Time_S4,Overall_processing_time,Overall_waiting_time,Tardiness,BREAKS,BREAKDOWN
0,12,1,34,SMD_0,0,402,402,AOI_2,402,781,...,0,CC_1,781,979,198,979,0,0,0,0
1,23,1,37,SMD_1,0,354,354,AOI_0,354,441,...,151,CC_1,592,696,104,696,0,0,0,0
2,131,1,29,SMD_3,0,37,37,AOI_0,37,90,...,48,0,0,0,0,138,0,0,0,0
3,29,1,18,SMD_0,402,1155,753,AOI_2,1155,1457,...,670,CC_0,2243,2441,198,1923,518,0,0,0
4,75,1,20,SMD_4,0,189,189,AOI_1,189,766,...,255,0,0,0,0,1021,0,0,0,0


In [ ]:
df.describe()

,ID,Priority,Family_type,Start_time_S1,Finish_time_S1,Processing_Time_S1,Start_time_S2,Finish_Time_S2,Processing_Time_S2,Start_time_S3,Finish_time_S3,Processing_Time_S3,Start_time_s4,Finish_time,Processing_Time_S4,Overall_processing_time,Overall_waiting_time,Tardiness,BREAKS,BREAKDOWN
count,17600.000000,17600.000000,17600.00000,17600.000000,17600.000000,17600.00000,17600.000000,17600.000000,17600.000000,17600.000000,17600.000000,17600.000000,17600.000000,17600.000000,17600.000000,17600.000000,17600.000000,17600.000000,17600.000000,17600.000000
mean,80.500000,10.212500,19.95000,7538.210795,8051.680966,513.47017,8152.902841,8580.564602,427.661761,7295.052045,7661.465682,366.413636,6578.648011,6721.836648,143.188636,1450.734205,7974.813182,45.258523,0.360909,0.299318
std,46.188432,5.233918,11.24354,4980.806574,5026.693312,443.46041,5091.827737,5181.531229,437.671922,5791.190746,5969.378769,401.553912,6279.250256,6365.629505,210.624005,1138.258964,5228.287285,225.555521,0.596289,0.457972
min,1.000000,1.000000,1.00000,0.000000,37.000000,37.00000,37.000000,90.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,126.000000,0.000000,0.000000,0.000000,0.000000
25%,40.750000,6.000000,11.00000,2750.000000,3281.250000,135.00000,3302.000000,4252.000000,87.000000,1518.000000,1938.000000,63.000000,0.000000,0.000000,0.000000,458.000000,2883.750000,0.000000,0.000000,0.000000
50%,80.500000,10.000000,20.00000,7172.500000,7643.000000,354.00000,7675.500000,8562.500000,241.000000,6641.000000,6929.500000,191.000000,5969.500000,6188.000000,70.000000,1033.000000,7628.500000,0.000000,0.000000,0.000000
75%,120.250000,15.000000,29.00000,11600.250000,12193.500000,869.00000,12493.000000,13361.000000,780.500000,12131.000000,13237.250000,657.000000,11489.000000,11651.000000,170.000000,2198.250000,12163.500000,0.000000,1.000000,1.000000
max,160.000000,19.000000,40.00000,17988.000000,18895.000000,1871.00000,18895.000000,19852.000000,1932.000000,19852.000000,20222.000000,1666.000000,21338.000000,21607.000000,1563.000000,5402.000000,20350.000000,3724.000000,3.000000,1.000000


In [ ]:
df['Anomaly_identification'] = (df['BREAKS'] > 0).astype(int)
print(f"Updated BREAKS distribution:\n{df['BREAKS'].value_counts().sort_index()}")

Updated BREAKS distribution:
BREAKS
0    12332
1     5268
Name: count, dtype: int64


In [ ]:
df.head()

,ID,Priority,Family_type,First_stage,Start_time_S1,Finish_time_S1,Processing_Time_S1,Second_stage,Start_time_S2,Finish_Time_S2,...,Fourth_stage,Start_time_s4,Finish_time,Processing_Time_S4,Overall_processing_time,Overall_waiting_time,Tardiness,BREAKS,BREAKDOWN,Anomaly_identification
0,12,1,34,SMD_0,0,402,402,AOI_2,402,781,...,CC_1,781,979,198,979,0,0,0,0,0
1,23,1,37,SMD_1,0,354,354,AOI_0,354,441,...,CC_1,592,696,104,696,0,0,0,0,0
2,131,1,29,SMD_3,0,37,37,AOI_0,37,90,...,0,0,0,0,138,0,0,0,0,0
3,29,1,18,SMD_0,402,1155,753,AOI_2,1155,1457,...,CC_0,2243,2441,198,1923,518,0,0,0,0
4,75,1,20,SMD_4,0,189,189,AOI_1,189,766,...,0,0,0,0,1021,0,0,0,0,0


In [ ]:
df["S1_efficiency"]    = df["Processing_Time_S1"] / (df["Finish_time_S1"] - df["Start_time_S1"] + 1)
df["S2_efficiency"]    = df["Processing_Time_S2"] / (df["Finish_Time_S2"] - df["Start_time_S2"] + 1)
df["S3_efficiency"]    = df["Processing_Time_S3"] / (df["Finish_time_S3"] - df["Start_time_S3"] + 1)
df["S4_efficiency"]    = df["Processing_Time_S4"] / (df["Finish_time"] - df["Start_time_s4"] + 1)

df["wait_to_process_ratio"] = df["Overall_waiting_time"] / (df["Overall_processing_time"] + 1)
df["longest_stage"]         = df[["Processing_Time_S1","Processing_Time_S2",
                                   "Processing_Time_S3","Processing_Time_S4"]].max(axis=1)
df["shortest_stage"]        = df[["Processing_Time_S1","Processing_Time_S2",
                                   "Processing_Time_S3","Processing_Time_S4"]].min(axis=1)
df["stage_time_std"]        = df[["Processing_Time_S1","Processing_Time_S2",
                                   "Processing_Time_S3","Processing_Time_S4"]].std(axis=1)
df["stage_time_range"]      = df["longest_stage"] - df["shortest_stage"]
df["is_tardy"]              = (df["Tardiness"] > 0).astype(int)
df["tardiness_per_unit"]    = df["Tardiness"] / (df["Overall_processing_time"] + 1)


In [ ]:
z_features = ["Overall_waiting_time", "Overall_processing_time", "Tardiness",
               "Processing_Time_S1","Processing_Time_S2","Processing_Time_S3","Processing_Time_S4"]
for col in z_features:
    df[f"zscore_{col}"] = np.abs(stats.zscore(df[col]))
    df[f"flag_z_{col}"] = (df[f"zscore_{col}"] > 3).astype(int)

df["total_z_flags"] = df[[f"flag_z_{c}" for c in z_features]].sum(axis=1)

print("\n=== ENGINEERED FEATURES — Sample ===")
eng_feats = ["S1_efficiency","S2_efficiency","S3_efficiency","S4_efficiency",
             "wait_to_process_ratio","stage_time_std","stage_time_range",
             "is_tardy","tardiness_per_unit","total_z_flags"]
print(df[eng_feats].describe().round(3).to_string())


=== ENGINEERED FEATURES — Sample ===
       S1_efficiency  S2_efficiency  S3_efficiency  S4_efficiency  wait_to_process_ratio  stage_time_std  stage_time_range   is_tardy  tardiness_per_unit  total_z_flags
count      17600.000      17600.000      17600.000      17600.000              17600.000       17600.000         17600.000  17600.000           17600.000      17600.000
mean           0.995          0.993          0.826          0.693                 11.218         266.458           591.179      0.064               0.021          0.072
std            0.005          0.007          0.372          0.454                 15.591         209.807           469.710      0.244               0.128          0.295
min            0.974          0.976          0.000          0.000                  0.000           4.272             9.000      0.000               0.000          0.000
25%            0.993          0.989          0.984          0.000                  2.535          80.333           17

In [ ]:
print("\n=== ENGINEERED FEATURES → BREAKDOWN CORRELATION ===")
for col in eng_feats:
    r, p = pointbiserialr(df["BREAKDOWN"], df[col].replace([np.inf, -np.inf], np.nan).fillna(0))
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    print(f"  {col:<30}: r={r:+.4f}  p={p:.4e}  {sig}")


=== ENGINEERED FEATURES → BREAKDOWN CORRELATION ===
  S1_efficiency                 : r=+0.3390  p=0.0000e+00  ***
  S2_efficiency                 : r=+0.3811  p=0.0000e+00  ***
  S3_efficiency                 : r=+0.1293  p=1.5657e-66  ***
  S4_efficiency                 : r=+0.0697  p=2.1023e-20  ***
  wait_to_process_ratio         : r=-0.2944  p=0.0000e+00  ***
  stage_time_std                : r=+0.7485  p=0.0000e+00  ***
  stage_time_range              : r=+0.7592  p=0.0000e+00  ***
  is_tardy                      : r=+0.2240  p=4.3495e-199  ***
  tardiness_per_unit            : r=+0.0916  p=4.2527e-34  ***
  total_z_flags                 : r=+0.3286  p=0.0000e+00  ***


In [ ]:
print("\n=== STEP 7 — STATISTICAL ANOMALY FLAGS ===")

key_feats = ["Overall_waiting_time", "Overall_processing_time", "Tardiness",
             "wait_to_process_ratio", "stage_time_std"]

# Z-Score method (|z| > 3)
z_scores = np.abs(stats.zscore(df[key_feats].fillna(0)))
z_anomaly = (z_scores > 3).any(axis=1).astype(int)

# IQR method
def iqr_flag(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return ((series < q1 - 1.5*iqr) | (series > q3 + 1.5*iqr)).astype(int)

iqr_flags = df[key_feats].apply(iqr_flag)
iqr_anomaly = iqr_flags.any(axis=1).astype(int)

print(f"Z-Score anomalies detected : {z_anomaly.sum():,} ({z_anomaly.mean()*100:.1f}%)")
print(f"IQR     anomalies detected : {iqr_anomaly.sum():,} ({iqr_anomaly.mean()*100:.1f}%)")
print(f"\nZ-Score vs BREAKDOWN:\n{pd.crosstab(z_anomaly, df['BREAKDOWN'], rownames=['Z-flag'], colnames=['BREAKDOWN'])}")
print(f"\nIQR vs BREAKDOWN:\n{pd.crosstab(iqr_anomaly, df['BREAKDOWN'], rownames=['IQR-flag'], colnames=['BREAKDOWN'])}")



=== STEP 7 — STATISTICAL ANOMALY FLAGS ===
Z-Score anomalies detected : 985 (5.6%)
IQR     anomalies detected : 2,841 (16.1%)

Z-Score vs BREAKDOWN:
BREAKDOWN      0     1
Z-flag                
0          11703  4912
1            629   356

IQR vs BREAKDOWN:
BREAKDOWN      0     1
IQR-flag              
0          10273  4486
1           2059   782
